# Week 65, Day 1: Profiling

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w065d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import cProfile, pstats

def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

profiler = cProfile.Profile()
profiler.enable()
fib(15)
profiler.disable()

stats = pstats.Stats(profiler).stats
for (filename, line, name), (primitive, total, *rest) in stats.items():
    if name == "fib":
        print(name, "called", total, "times;", primitive, "of them from outside")

In [ ]:
import time

def slow_unique(items):
    seen = []
    for item in items:
        if item not in seen:          # a hidden loop: 'in' on a list checks every item
            seen.append(item)
    return seen

def fast_unique(items):
    return list(dict.fromkeys(items))  # keeps first occurrences, in order

data = list(range(2000)) * 2
for func in (slow_unique, fast_unique):
    start = time.perf_counter()
    result = func(data)
    elapsed = time.perf_counter() - start
    print(func.__name__, len(result), "items", "fast" if elapsed < 0.05 else "slow")
print(slow_unique(data) == fast_unique(data))

In [ ]:
import timeit

setup = "data = list(range(1000)); as_set = set(data)"
for stmt in ("999 in data", "999 in as_set"):
    best = min(timeit.repeat(stmt, setup=setup, number=2000, repeat=3)) / 2000
    print(f"{stmt:15} {'under' if best < 1e-4 else 'over'} 0.1 ms per run")

def squares_loop(n):
    out = []
    for i in range(n):
        out.append(i * i)
    return out

def squares_comp(n):
    return [i * i for i in range(n)]

t_loop = min(timeit.repeat(lambda: squares_loop(1000), number=200, repeat=3))
t_comp = min(timeit.repeat(lambda: squares_comp(1000), number=200, repeat=3))
print(squares_loop(10) == squares_comp(10), "both timed:", t_loop > 0 and t_comp > 0)

In [ ]:
import cProfile, io, pstats

def parse(line):
    name, qty, price = line.split(",")
    return name, int(qty), float(price)

def total(lines):
    return sum(qty * price for _, qty, price in map(parse, lines))

lines = ["tea,2,1.5", "jam,1,3.25"] * 500
profiler = cProfile.Profile()
profiler.enable()
result = total(lines)
profiler.disable()

out = io.StringIO()
pstats.Stats(profiler, stream=out).sort_stats("ncalls").print_stats(3)
report = out.getvalue()
print(result, "ncalls" in report, "parse" in report)
calls = {key[2]: value[1] for key, value in pstats.Stats(profiler).stats.items()}
print(calls["parse"], calls["total"])

In [ ]:
import cProfile, pstats

def is_prime(n):
    return n > 1 and all(n % d for d in range(2, int(n ** 0.5) + 1))

def primes_upto(n):
    return [k for k in range(n) if is_prime(k)]

profiler = cProfile.Profile()
profiler.enable()
primes_upto(3000)
profiler.disable()

rows = []
for (file, line, name), (cc, nc, tt, ct, callers) in pstats.Stats(profiler).stats.items():
    if name in ("is_prime", "primes_upto"):
        rows.append((name, nc, ct >= tt))
print(sorted(rows))

In [ ]:
import tracemalloc

def peak_of(func):
    tracemalloc.start()
    tracemalloc.reset_peak()
    func()
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak

as_list = peak_of(lambda: sum([i * i for i in range(200000)]))
as_stream = peak_of(lambda: sum(i * i for i in range(200000)))
print(as_list > 10 * as_stream)

tracemalloc.start()
before = tracemalloc.take_snapshot()
cache = [str(i) * 10 for i in range(5000)]
after = tracemalloc.take_snapshot()
top = after.compare_to(before, "lineno")[0]
print(top.size_diff > 0, top.count_diff >= 5000)
tracemalloc.stop()

> **Key idea.** Optimise only once code works and only where it matters; decide what "fast enough" means, then measure, profile, fix the biggest cost, check results and measure again. Use time.perf_counter for durations; use timeit with several repeats and take the minimum to compare small pieces fairly, on the same data. cProfile records every call: pstats reports ncalls (total/primitive, revealing recursion), tottime (time in the function itself) and cumtime (including callees); sort by cumtime to find the expensive step and by tottime to find slow code, and treat profiled times as relative. tracemalloc measures current and peak memory and, with snapshots, which lines allocated it.

### Exercise: A timing context manager

Write a context manager class `Timer(clock=time.perf_counter, label="")` that records the time when the block starts and ends, using the given clock: `elapsed` is `None` while the block is still running and the duration afterwards, even if the block raised (the exception still propagates). A `Timer` can be reused; each use replaces `elapsed`. `str(timer)` is like `"load: 0.250s"` (three decimal places; just `"0.250s"` with no label). The tests pass a fake clock, so the numbers are exact.

In [ ]:
import time

class Timer:
    def __init__(self, clock=time.perf_counter, label=""):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timer():\n    \"timing with a clock\"\n    ticks = iter([1.0, 1.25, 5.0, 7.5, 10.0, 10.125])\n    t = Timer(clock=lambda: next(ticks), label=\"load\")\n    with t as same:\n        assert same is t and t.elapsed is None\n    assert t.elapsed == 0.25 and str(t) == \"load: 0.250s\"\n    with t:\n        pass\n    assert t.elapsed == 2.5\n    try:\n        with t:\n            raise ValueError\n        assert False\n    except ValueError:\n        pass\n    assert t.elapsed == 0.125\n    plain = Timer(clock=iter([0.0, 2.0]).__next__)\n    with plain:\n        pass\n    assert str(plain) == \"2.000s\"\n    real = Timer()\n    with real:\n        sum(range(1000))\n    assert real.elapsed >= 0")

### Exercise: Fair comparisons

Write `best_time(func, number=100, repeat=5, timer=None)`: the best (smallest) time **per call** of `func()` over `repeat` rounds of `number` calls each, using `timeit.repeat` (pass `timer` on to it when given, so tests can use a fake clock). Then `compare(funcs, *args, number=100)`: check that every function in the dict `funcs` (name to function) returns the **same result** for `args` (raising `ValueError("results differ")` otherwise), then return the names sorted from fastest to slowest by `best_time`.

In [ ]:
import timeit

def best_time(func, number=100, repeat=5, timer=None):
    ___

def compare(funcs, *args, number=100):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timeit():\n    \"timeit\"\n    ticks = iter([0, 10, 0, 4, 0, 7])\n    assert best_time(lambda: None, number=2, repeat=3, timer=lambda: next(ticks)) == 2.0\n    assert best_time(lambda: sum(range(100))) >= 0\n    def slow(n):\n        return sorted(set(range(n)) | set(range(n)))[-1]\n    def fast(n):\n        return n - 1\n    assert compare({\"slow\": slow, \"fast\": fast}, 3000, number=20) == [\"fast\", \"slow\"]\n    try:\n        compare({\"a\": lambda n: n, \"b\": lambda n: n + 1}, 3)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"results differ\"")

### Exercise: Counting calls with cProfile

Write `call_counts(func, *args)`: profile one call of `func(*args)` with `cProfile` and return `(result, counts)`, where `counts` is a dict from function name to total number of calls, for functions defined in the **same file** as `func` only (compare each entry's filename with `func.__code__.co_filename`). Then `recursive(func, *args)`: the sorted names of those functions whose primitive call count is smaller than their total (they called themselves).

In [ ]:
import cProfile
import pstats

def call_counts(func, *args):
    ___

def recursive(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_calls():\n    \"cProfile and pstats\"\n    def fib(n):\n        return n if n < 2 else fib(n - 1) + fib(n - 2)\n    def helper(x):\n        return x * 2\n    def main(n):\n        return sum(helper(i) for i in range(n)) + fib(5)\n    result, counts = call_counts(main, 4)\n    assert result == 17 and counts[\"main\"] == 1 and counts[\"helper\"] == 4 and counts[\"fib\"] == 15\n    assert recursive(main, 4) == [\"fib\"] and recursive(helper, 1) == []\n    assert all(isinstance(name, str) for name in counts)")

### Exercise: Reading a report

Profile data is just numbers. Given rows as dicts with `name`, `ncalls`, `tottime` and `cumtime`, write `top(rows, key, n=3)`: the names of the `n` rows with the largest value of `key`, largest first (ties: alphabetical). Then `diagnose(row)`: `"slow body"` if `tottime` is at least half of `cumtime`, otherwise `"slow callees"`; but first, `"too many calls"` if `ncalls` is at least 10,000 and the time per call (`tottime / ncalls`) is under a microsecond.

In [ ]:
def top(rows, key, n=3):
    ___

def diagnose(row):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"interpreting profiles\"\n    rows = [\n        {\"name\": \"main\", \"ncalls\": 1, \"tottime\": 0.01, \"cumtime\": 2.0},\n        {\"name\": \"parse\", \"ncalls\": 50000, \"tottime\": 0.04, \"cumtime\": 0.05},\n        {\"name\": \"load\", \"ncalls\": 3, \"tottime\": 1.2, \"cumtime\": 1.3},\n        {\"name\": \"save\", \"ncalls\": 3, \"tottime\": 1.2, \"cumtime\": 1.25},\n    ]\n    assert top(rows, \"cumtime\") == [\"main\", \"load\", \"save\"] and top(rows, \"tottime\", 2) == [\"load\", \"save\"]\n    assert top(rows, \"ncalls\", 1) == [\"parse\"]\n    assert [diagnose(r) for r in rows] == [\"slow callees\", \"too many calls\", \"slow body\", \"slow body\"]")

### Exercise: Measuring memory

Write `peak_memory(func, *args)`: run `func(*args)` with `tracemalloc` tracking and return `(result, peak)`, the peak bytes allocated **during the call** (reset the peak first). If tracing was already on before the call, leave it on afterwards; otherwise stop it. Then `lighter(funcs, *args)`: the name (from the dict `funcs`) of the function with the smallest peak.

In [ ]:
import tracemalloc

def peak_memory(func, *args):
    ___

def lighter(funcs, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memory():\n    \"tracemalloc\"\n    import tracemalloc\n    def as_list(n):\n        return sum([i * i for i in range(n)])\n    def as_stream(n):\n        return sum(i * i for i in range(n))\n    r1, p1 = peak_memory(as_list, 100000)\n    r2, p2 = peak_memory(as_stream, 100000)\n    assert r1 == r2 and p1 > 10 * p2 and not tracemalloc.is_tracing()\n    assert lighter({\"list\": as_list, \"stream\": as_stream}, 50000) == \"stream\"\n    tracemalloc.start()\n    try:\n        peak_memory(as_stream, 10)\n        assert tracemalloc.is_tracing()\n    finally:\n        tracemalloc.stop()")

### Bug hunt: Timing the setup too

The timer starts **before** the data is built, so the measurement includes building the list, and the reported time is the setup plus the search: with this fake clock it prints `0.9`. Start timing after the setup, just around the search, so it prints `0.2`.

In [ ]:
ticks = iter([1.0, 1.7, 1.9])
clock = lambda: next(ticks)

start = clock()
data = list(range(100000))
setup_done = clock()
found = 99999 in data
end = clock()
print(round(end - start, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"time the right part\"\n    expect_output(\"0.2\")")

### Bug hunt: The average of noisy runs

These five timings include one run that was interrupted by something else on the machine. Taking the **mean** lets that one run distort the result, and this prints `0.28`. Report the **minimum**, the best estimate of the code's real cost, so it prints `0.1`.

In [ ]:
runs = [0.11, 0.10, 0.12, 0.97, 0.10]
print(round(sum(runs) / len(runs), 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the minimum\"\n    expect_output(\"0.1\")")

### Bug hunt: A profile of the wrong thing

The profiler is disabled before the function runs, so it records almost nothing and `parse` doesn't appear: this prints `0`. Enable it around the call so the count of `parse` calls is recorded, and this prints `3`.

In [ ]:
import cProfile, pstats

def parse(text):
    return int(text)

profiler = cProfile.Profile()
profiler.enable()
profiler.disable()
values = [parse(t) for t in ("1", "2", "3")]
counts = {k[2]: v[1] for k, v in pstats.Stats(profiler).stats.items()}
print(counts.get("parse", 0))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"profile the call\"\n    expect_output(\"3\")")

### Puzzle: Profiling a call

Arrange the lines of a function that profiles a call and prints the ten most expensive functions by cumulative time. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the clock to use for measuring durations: "time.time" or "time.perf_counter"
q = ___   # which statistic of repeated timings to report: "mean" or "min"
r = ___   # the pstats column that excludes time in called functions: "tottime" or "cumtime"
s = ___   # ncalls shows "40/1": how many calls were primitive (not recursive)?
t = ___   # True or False: timeit's setup code is included in the time

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"time.perf_counter\" and q == \"min\" and r == \"tottime\", \"p, q and r\"\n    assert s == 1 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a deterministic profiler

Write your own profiler using `sys.settrace` (Week 64) and a clock you can choose. `Profiler(clock=time.perf_counter)` is a context manager that traces calls of Python functions while the block runs (restoring the previous tracer afterwards, even on errors), and records, per function name:

- `ncalls`: the number of calls;
- `tottime`: time spent in the function's own code (from its call to its return, **minus** time spent in functions it called);
- `cumtime`: time from call to return, **counting recursion only once**: when a function is already running further down the stack, a nested call's time isn't added to its `cumtime` again.

`stats()` returns a dict from name to a dict with those three keys. `ranked(key)` returns names sorted by that key, largest first (ties: alphabetical). Read the clock exactly **once** per `"call"` event and once per `"return"` event, and nowhere else, so that a fake clock gives exact results. Ignore the context manager's own functions.

In [ ]:
import sys
import time

class Profiler:
    def __init__(self, clock=time.perf_counter):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_profiler():\n    \"a deterministic profiler\"\n    t = [0]\n    def clock():\n        t[0] += 1\n        return t[0]\n    def leaf():\n        return 1\n    def middle():\n        return leaf() + leaf()\n    def top_level():\n        return middle() + leaf()\n    with Profiler(clock) as p:\n        top_level()\n    s = p.stats()\n    assert s[\"leaf\"] == {\"ncalls\": 3, \"tottime\": 3, \"cumtime\": 3}\n    assert s[\"middle\"] == {\"ncalls\": 1, \"tottime\": 3, \"cumtime\": 5}\n    assert s[\"top_level\"] == {\"ncalls\": 1, \"tottime\": 3, \"cumtime\": 9}\n    assert p.ranked(\"cumtime\") == [\"top_level\", \"middle\", \"leaf\"] and p.ranked(\"tottime\") == [\"leaf\", \"middle\", \"top_level\"]\n\ndef test_recursion_and_errors():\n    \"recursion counted once; errors\"\n    import sys\n    t = [0]\n    def clock():\n        t[0] += 1\n        return t[0]\n    def down(n):\n        return 0 if n == 0 else down(n - 1)\n    before = sys.gettrace()\n    with Profiler(clock) as p:\n        down(2)\n    s = p.stats()[\"down\"]\n    assert s[\"ncalls\"] == 3 and s[\"cumtime\"] == 5 and s[\"tottime\"] == 5\n    def boom():\n        raise ValueError\n    try:\n        with Profiler(clock) as q:\n            boom()\n        assert False\n    except ValueError:\n        pass\n    assert q.stats()[\"boom\"][\"ncalls\"] == 1 and sys.gettrace() is before")

## Warm-up

`fib(15)` looks like one call, but it makes nearly two thousand. A profiler counts every call and the time spent in each function, and that's how you find out where a program really spends its time: often somewhere you'd never have guessed.

## Measure first

Three rules, in order:

1. **Make it work, then make it right, then make it fast** (a saying usually credited to Kent Beck). Fast code that gives wrong answers is worthless, and clear code is easier to speed up later.
2. **Only optimise what matters.** If a script runs once a week in two seconds, leave it alone. Decide what "fast enough" means (a page in under 200 milliseconds, a nightly job in under an hour) before you start.
3. **Measure, don't guess.** Programs usually spend most of their time in a small part of the code (the "80/20" pattern). Speeding up anything else is wasted effort, and the slow part is very often not where you expect: a hidden loop inside a helper, a lookup done a million times, or reading a file again and again.

The process is a loop: measure the whole program, **profile** to find the hot spot, fix the biggest cost (often with a better algorithm, tomorrow's subject), check that results are unchanged (your tests from Week 24), measure again, and stop when it's fast enough.

## Timing with perf_counter and timeit

`time.perf_counter()` is the right clock for measuring durations: the highest resolution available, and never adjusted (unlike `time.time()`, which can jump when the system clock is corrected). Subtract two readings to get seconds.

A single timing is noisy: other programs, caches warming up, and the garbage collector all interfere. For small pieces of code, use **`timeit`**:

- `timeit.timeit(stmt, setup="", number=1000000, globals=None)` runs the statement `number` times and returns the **total** time; divide by `number` for one run. `setup` runs once beforehand and isn't timed.
- `timeit.repeat(..., repeat=5)` does that several times and returns a list. Take the **minimum**: the noise only ever makes things slower, so the minimum is the closest to the true cost. (The mean is misleading when a few runs were interrupted.)
- `timeit` switches off the garbage collector while timing, for consistency.
- From a terminal, `python -m timeit -s "setup" "statement"` chooses `number` for you and prints a summary.

Compare alternatives **in the same run**, on the same data, and check that they give the same results.

## Profiling with cProfile

Timing tells you **how long**; a **profiler** tells you **where**. `cProfile` (written in C, so its overhead is moderate) records every function call and return, counting calls and adding up time per function:

- `cProfile.run("main()")` profiles a statement and prints a report;
- `profiler = cProfile.Profile()`, then `profiler.enable()` ... `profiler.disable()` (or `with cProfile.Profile() as profiler:` in 3.8+) profiles a section of a program;
- `python -m cProfile -s cumulative script.py` profiles a whole script from the terminal, sorted by cumulative time.

Reports are produced by **`pstats.Stats(profiler)`**: `.sort_stats("cumulative")` (or `"tottime"`, `"ncalls"`), then `.print_stats(10)` for the top ten lines. The underlying data is `stats.stats`, a dict keyed by `(filename, line, function name)` whose values start with `(primitive calls, total calls, tottime, cumtime, callers)`.

Profilers slow programs down (every call is recorded), and they slow small, frequently called functions most, so treat the times as **relative**, and confirm a suspected hot spot with `timeit`.

## Reading a profile

A `pstats` report has one line per function with these columns:

- **ncalls**: how many times it was called. Two numbers like `177/1` mean 177 calls in total, 1 of them **primitive** (not recursive): recursion shows up this way.
- **tottime**: time spent **in the function itself**, excluding the functions it called. Large `tottime` means the function's own code is slow.
- **percall**: `tottime / ncalls`.
- **cumtime**: time spent in the function **and everything it called**. Large `cumtime` with small `tottime` means it's a manager: the cost is further down.
- **filename:lineno(function)**: which function. Built-ins appear as `{built-in method ...}` or `{method 'append' of 'list' objects}`.

How to read one: sort by **cumtime** to see the big picture (which top-level step costs the most), then follow the costs down to functions with high **tottime**. Very high **ncalls** with small per-call time is a sign to reduce the **number** of calls (cache, batch, or change the algorithm) rather than to speed up each one.

## Measuring memory

Memory matters too: a program that needs more RAM than the machine has slows to a crawl or crashes. **`tracemalloc`** tracks every memory block Python allocates:

- `tracemalloc.start()` begins tracking (it slows the program, like profiling); `tracemalloc.stop()` ends it.
- `tracemalloc.get_traced_memory()` returns `(current, peak)` in bytes since starting; `tracemalloc.reset_peak()` (3.9+) restarts the peak.
- `tracemalloc.take_snapshot()` records where memory was allocated; `snapshot.statistics("lineno")` lists the source lines responsible, biggest first, and `snapshot2.compare_to(snapshot1, "lineno")` shows what grew: the way to hunt leaks (Week 62).

The usual memory wins: process data as a **stream** with generators (Week 59) instead of building whole lists, use compact types (tomorrow and Wednesday), and drop references to large objects you no longer need.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run `python -m cProfile -s cumulative` on a script you've written and find its most expensive function. If you'd like a picture, the free tool `snakeviz` draws the profile as a chart. Tomorrow: why the algorithm usually matters more than anything else, and when constant factors win.